# Espectroscopía de dos tonos con flujo (bobina) y drive por la antena

Para cada corriente de la bobina:
1. la fuente rampa hasta la corriente y espera;
2. con el **drive apagado**, el VNA barre alrededor del resonador y el **marcador busca el mínimo de |S21|**
   (`FUNC MIN` con tracking sobre la traza S21/MLOG); con `CALC:MEAS:MARK:SET CWFReq` el VNA copia la frecuencia
   del marcador a su **frecuencia CW**;
3. el VNA pasa a **CW** en esa frecuencia y se mide una referencia sin drive;
4. se enciende el SG y se barre el drive (potencia × frecuencia) leyendo el VNA en CW.

El mínimo se busca **solo al cambiar la corriente de la bobina**, nunca al cambiar la frecuencia o la potencia del drive.
Cada corriente es un run propio de QCoDeS (con su `run_id` y su metadata) en la base de la carpeta `two_tone_flux2`.
Funciones en `dos_tonos.py`.

In [ ]:
# --- Rutas del repositorio (celda agregada al subir a GitHub) ---
# El código está en ../codigo y las bases de datos en ../datos. Esta celda agrega ../codigo al path
# y se para en ../datos, así las rutas relativas de las bases (p. ej. 'fluxonium_2/punch_out.db')
# funcionan igual que en la PC del laboratorio. Se puede correr más de una vez.
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'codigo').is_dir() and (p / 'datos').is_dir())
if str(REPO / 'codigo') not in sys.path:
    sys.path.insert(0, str(REPO / 'codigo'))
os.chdir(REPO / 'datos')
print('repo:', REPO, '| cwd:', os.getcwd())

In [1]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import qcodes as qc
from qcodes.dataset import initialise_or_create_database_at, load_or_create_experiment, load_by_id
from qcodes.instrument_drivers.Keysight import KeysightN5245A
from qcodes.instrument_drivers.Keysight.Keysight_N5173B import KeysightN5173B

from barrido_flujo import conectar_fuente
from dos_tonos import medir_dos_tonos, tiempo_estimado, cargar_dos_tonos, buscar_dos_tonos

%matplotlib tk

### Base de datos

In [2]:
carpeta_db = os.path.abspath('two_tone_flux2')
os.makedirs(carpeta_db, exist_ok=True)

path2database = os.path.join(carpeta_db, 'two_tone_flux2.db')
initialise_or_create_database_at(path2database)
exp = load_or_create_experiment(experiment_name='dos_tonos', sample_name='fluxonium')
print('base:', qc.config.core.db_location)
print(exp)

base: c:\Users\Leandro\Documents\RF\rf-2026\qubits\two_tone_flux2\two_tone_flux2.db
dos_tonos#fluxonium#1@c:\Users\Leandro\Documents\RF\rf-2026\qubits\two_tone_flux2\two_tone_flux2.db
---------------------------------------------------------------------------------------------------
1-dos_tonos_Q4_I-139.40uA-1-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-13468
2-dos_tonos_Q4_I-139.40uA-2-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-13468
3-dos_tonos_Q4_I-139.40uA-3-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-648
4-dos_tonos_Q4_I-139.40uA-4-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-10098
5-dos_tonos_Q4_I-139.40uA-5-p_drive,f_drive,f_busqueda,s21_re,s21_im,s21_mag,s21_phase,s21_err,busqueda_re,busqueda_im,busqueda_mag-7038
6-dos_tonos_Q4_I-139.40uA-6-p

### Conexión: PNA, generador (drive por la antena) y fuente de la bobina
Si otro notebook (`punch_out.ipynb`, `readout.ipynb`) tiene abiertos el PNA, el SG o la Keithley, cerralos ahí primero.
`I_max` es un límite de seguridad: la fuente no acepta corrientes mayores.

In [3]:
addr_pna = 'TCPIP::192.168.1.6::inst0::INSTR'
pna = KeysightN5245A('pna', address=addr_pna)

sg = KeysightN5173B('sg', address='GPIB1::19::INSTR')
sg.rf_output(False)                                             # apagado hasta que empiece la medición

station = qc.Station(pna, sg)

fuente = conectar_fuente('GPIB0::15::INSTR', I_max=300e-6)     # A
print(pna.ask('*IDN?').strip())
print(sg.ask('*IDN?').strip(), f'| f = {sg.frequency()/1e9:.6f} GHz, P = {sg.power():.2f} dBm, salida {sg.rf_output()}')
print(f'corriente actual de la bobina: {fuente.actual() * 1e6:.3f} uA')

Connected to: Keysight Technologies N5242B (serial:MY63056409, firmware:A.15.71.08) in 5.00s
Connected to: Agilent Technologies N5173B (serial:MY62220514, firmware:B.01.96) in 0.39s


[pna_port3(KeysightPNAPort)] Snapshot: Could not update parameter: source_power
[pna_port4(KeysightPNAPort)] Snapshot: Could not update parameter: source_power
[pna_port3(KeysightPNAPort)] Snapshot: Could not update parameter: source_power
[pna_port4(KeysightPNAPort)] Snapshot: Could not update parameter: source_power


Keysight Technologies,N5242B,MY63056409,A.15.71.08
Agilent Technologies, N5173B, MY62220514, B.01.96 | f = 3.996000 GHz, P = 10.00 dBm, salida False
corriente actual de la bobina: 0.000 uA


### Parámetros
- **Bobina**: corrientes a medir (un run por corriente). Q4 tiene histéresis: el estado n = 0 se prepara rampeando
  hasta −200 µA y subiendo (`preparar_desde`).
- **Búsqueda del mínimo**: barrido LIN del VNA con el drive apagado; el marcador deja la f CW en el mínimo.
  El VNA necesita una traza **S21 en formato MLOG** (sobre esa busca el marcador 1).
- **Lectura CW**: `puntos_cw` puntos promediados en cada frecuencia del drive (`puntos_cw / ifbw_cw` s por punto).
- **Drive**: frecuencias y potencias del SG (las potencias son el lazo externo).

In [6]:
# ---- bobina (flujo) ----
corrientes     = np.array([-0.805, -0.005, -0.405, -0.605, -0.205,
                           -0.805, -0.005, -0.405, -0.205, -0.605]) * 1e-6   # A, 5 corrientes x 2, desordenadas
preparar_desde = 0                     # la bobina quedó en -0.805 uA, que es la primera
paso_rampa     = 0.005e-6                 # A
espera_I       = 0.25                      # s
volver_a       = None

# ---- VNA: búsqueda del mínimo (una sola vez, al principio) ----
centro_busqueda = 5.99e9
span_busqueda   = 30e6
puntos_busqueda = 201
ifbw_busqueda   = 10
seguir_minimo   = True
buscar_cada     = 0                       # f CW fija toda la prueba

# ---- VNA: lectura CW ----
p_vna             = 0                     # dBm, como en 681-713
puntos_cw         = 30                    # 3 s por f de drive -> ~1.1 kHz por punto (antes ~2 kHz)
ifbw_cw           = 10
desplazamiento_cw = 2.1e6                 # f CW en fr, como en 681-713

# ---- drive (SG por la antena) ----
f_drive = np.concatenate([np.arange(2.751e9, 2.8811e9, 5e6),    # línea de 2.81 GHz con sus bordes (2.796 y 2.826)
                          np.arange(2.951e9, 3.0511e9, 5e6)])   # f_CW/2 = 2.996 GHz y alrededores
p_drive      = [10]                       # dBm, como en 681-713
espera_drive = 0.01
excluir_cw   = 20e6
n_ref        = 5

# ---- metadata ----
resonador = 'res_q4'
qubit     = 'Q4'
att_vna   = -62 - 50
att_drive = '20'
notas     = ('prueba I vs tiempo: 2.75-2.88 y 2.95-3.05 GHz, corrientes alternadas (2 pasadas), '
             'VNA 0 dBm, 30 pts CW a 10 Hz, f CW en fr, drive +10 dBm')

t_tot, t_pt = tiempo_estimado(len(corrientes), len(p_drive), len(f_drive), puntos_cw, ifbw_cw, espera_drive,
                              puntos_busqueda, ifbw_busqueda, espera_I, n_ref, 0.05, buscar_cada)
print(f'{len(corrientes)} corriente(s) x {len(f_drive)} frecuencias | ~{t_pt*1e3:.0f} ms por punto -> ~{t_tot/60:.1f} min')

10 corriente(s) x 48 frecuencias | ~3060 ms por punto -> ~27.4 min


## Medición en vivo
Se abre una ventana (matplotlib tk) que se actualiza mientras se mide:
- arriba: la traza de búsqueda con el mínimo y la f CW; la f CW en cada corriente;
- medio: Δ|S21| y Δfase respecto de la referencia sin drive, vs la frecuencia del drive;
- abajo: los mapas (eje y = corriente de la bobina si hay más de una, si no la potencia del drive).

Para cortar: interrumpí el kernel (■). Lo medido queda guardado, el SG se apaga y el VNA vuelve a barrer el resonador.

In [7]:
res = medir_dos_tonos(
    pna, sg, fuente, corrientes, f_drive, p_drive,
    centro_busqueda=centro_busqueda, span_busqueda=span_busqueda,
    puntos_busqueda=puntos_busqueda, ifbw_busqueda=ifbw_busqueda, seguir_minimo=seguir_minimo,
    p_vna=p_vna, puntos_cw=puntos_cw, ifbw_cw=ifbw_cw, desplazamiento_cw=desplazamiento_cw,
    espera_drive=espera_drive, excluir_cw=excluir_cw, n_ref=n_ref,
    paso_rampa=paso_rampa, espera_I=espera_I, preparar_desde=preparar_desde, volver_a=volver_a,
    vivo=True, nombre=f'dos_tonos_{qubit}',
    resonador=resonador, qubit=qubit, att_vna=att_vna, att_drive=att_drive, notas=notas)
runs_dt = [r['run_id'] for r in res['runs']]
print('runs:', runs_dt)

dos tonos 20261113_142016: 10 corriente(s) x 1 potencia(s) x 48 frecuencias del drive, 1 búsqueda(s) del mínimo | ~3060 ms por punto -> ~27.4 min (sin la rampa de la bobina)
preparando: rampa a 0.00 uA
I =   -0.805 uA | mínimo 5.989550 GHz (-71.26 dB) | f CW 5.991650 GHz | ref -69.72 dB
Starting experimental run with id: 714. 
  run 714 listo
I =   -0.005 uA | f CW fija 5.991650 GHz | ref -69.71 dB, +60.02°
Starting experimental run with id: 715. 
  run 715 listo
I =   -0.405 uA | f CW fija 5.991650 GHz | ref -69.71 dB, +59.92°
Starting experimental run with id: 716. 
  run 716 listo
I =   -0.605 uA | f CW fija 5.991650 GHz | ref -69.71 dB, +59.89°
Starting experimental run with id: 717. 
  run 717 listo
I =   -0.205 uA | f CW fija 5.991650 GHz | ref -69.70 dB, +59.93°
Starting experimental run with id: 718. 
  run 718 listo
I =   -0.805 uA | f CW fija 5.991650 GHz | ref -69.70 dB, +59.78°
Starting experimental run with id: 719. 
  run 719 listo
I =   -0.005 uA | f CW fija 5.991650 GHz

In [12]:
# ---- Graficar runs de dos tonos: Δ|S21| y Δfase (I bobina en x, f drive en y) ----
import numpy as np
import matplotlib.pyplot as plt
from dos_tonos import cargar_dos_tonos, buscar_dos_tonos

tag_graf     = '20261112_114816'   # tag de la medición (lo imprime medir_dos_tonos); None para usar runs_graf
runs_graf    = None                # p.ej. list(range(681, 714)); se usa si tag_graf es None
referencia   = 'sin_drive'         # 'sin_drive' = como la ventana en vivo (ref. sin drive de cada corriente)
                                   # 'mediana'   = mediana del propio barrido (saca la deriva entre corrientes)
restar_fondo = False               # True: resta la mediana de todos los runs en cada f (saca las líneas fijas)
f_zona       = None                # (fmin, fmax) en GHz para hacer zoom; None = todo el barrido
escala       = 3.0                 # límites de color = ± escala × ruido

# ---- carga ----
ids = [r['run_id'] for r in buscar_dos_tonos(tag_graf)['runs']] if tag_graf else list(runs_graf)
D = [cargar_dos_tonos(r) for r in ids]
F = D[0]['f_drive']
I = np.array([d['metadata']['current_uA'] for d in D])
print(f"{len(ids)} runs ({ids[0]}-{ids[-1]}), I de {I.min():+.3f} a {I.max():+.3f} uA, {len(F)} f de drive")

MAG = np.full((len(D), len(F)), np.nan); FAS = np.full_like(MAG, np.nan)
for i, d in enumerate(D):
    z = np.full(len(F), np.nan + 0j)
    k = np.searchsorted(F, d['f_drive']); ok = (k < len(F)) & (F[np.minimum(k, len(F) - 1)] == d['f_drive'])
    z[k[ok]] = d['z'][ok]
    ref = d['z_ref'] if referencia == 'sin_drive' else np.nanmedian(z.real) + 1j * np.nanmedian(z.imag)
    MAG[i] = 20 * np.log10(np.abs(z / ref))           # Δ|S21| [dB]
    FAS[i] = np.degrees(np.angle(z / ref))            # Δfase [°]

# corrientes repetidas (ida y vuelta) -> promedio
Iu = np.unique(np.round(I, 6))
prom = lambda Y: np.array([np.nanmean(Y[np.isclose(I, x)], 0) for x in Iu])
MAGu, FASu = prom(MAG), prom(FAS)
if restar_fondo:
    MAGu, FASu = MAGu - np.nanmedian(MAGu, 0), FASu - np.nanmedian(FASu, 0)
rob = lambda x: 1.4826 * np.nanmedian(np.abs(x - np.nanmedian(x)))
sel = np.ones(len(F), bool) if f_zona is None else (F / 1e9 >= f_zona[0]) & (F / 1e9 <= f_zona[1])

# ---- figura ----
fig, ax = plt.subplots(2, 2, figsize=(15, 10), gridspec_kw=dict(height_ratios=[1.6, 1]))
dI = np.median(np.abs(np.diff(Iu))) if len(Iu) > 1 else 0.01
ext = [Iu.min() - dI / 2, Iu.max() + dI / 2, F[sel][0] / 1e9, F[sel][-1] / 1e9]
cm = plt.get_cmap('RdBu_r').copy(); cm.set_bad('0.85')
for c, (M, Y, nom, u) in enumerate(((MAGu, MAG, 'Δ|S21|', 'dB'), (FASu, FAS, 'Δfase', '°'))):
    a = ax[0, c]
    v = escala * rob(M[:, sel])
    im = a.imshow(M[:, sel].T, aspect='auto', origin='lower', cmap=cm, vmin=-v, vmax=v,
                  extent=ext, interpolation='nearest')
    a.set_xlabel('I bobina [uA]'); a.set_ylabel('f drive [GHz]')
    a.set_title(f"{nom} [{u}]{' − fondo común' if restar_fondo else ''} (ref.: {referencia.replace('_', ' ')})")
    fig.colorbar(im, ax=a, label=f'{nom} [{u}]')
    a = ax[1, c]                                    # espectro promedio sobre todas las corrientes
    m = np.nanmean(Y, 0)[sel]
    a.plot(F[sel] / 1e9, m, lw=.7, color='C0' if c == 0 else 'C3')
    a.axhline(0, color='0.6', lw=.8)
    a.set_xlabel('f drive [GHz]'); a.set_ylabel(f'{nom} [{u}]')
    a.set_title(f'{nom} promedio de los {len(ids)} runs (ruido por run {rob(Y[:, sel]):.3g} {u})')
    a.set_xlim(ext[2], ext[3]); a.grid(alpha=.25)
fig.suptitle(f"dos tonos {tag_graf or ''}: runs {ids[0]}-{ids[-1]}")
fig.tight_layout()
plt.show()

33 runs (681-713), I de -0.805 a -0.005 uA, 800 f de drive


## Un tono: barrido de corriente leyendo en CW (sin drive)
El SG queda apagado y solo se barre la corriente de la bobina; en cada corriente el VNA lee S21 en CW
(`puntos_cw_1t` puntos promediados). Todo el barrido es **un solo run** con setpoint `I_bobina`.

Cómo se elige la f CW:
- `buscar_cada_1t = 0`: busca el mínimo solo en la primera corriente y deja la f CW fija (el corrimiento del
  resonador con el flujo se ve como cambio de fase/amplitud en CW);
- `buscar_cada_1t = n`: vuelve a buscar el mínimo cada n corrientes (1 = en todas, la f CW sigue al resonador);
- `f_cw_1t = ...` [Hz]: f CW fija en ese valor, sin buscar nunca.

La ventana en vivo muestra la búsqueda, el mínimo vs I, y |S21| y fase en CW vs I. Para cortar: ■ (lo medido queda guardado).

In [17]:
import importlib, dos_tonos
importlib.reload(dos_tonos)                  # por si el kernel ya tenía cargada la versión vieja
from dos_tonos import medir_un_tono, cargar_un_tono, tiempo_estimado_un_tono

# ---- bobina (flujo) ----
corrientes_1t     = np.arange(-0.005e-6, -2.0e-6, -0.005e-6)   # A (10 nA ~ 0.015 Φ0 si el período es ~0.69 uA)
preparar_desde_1t = -1.995e-6                  # A, rampa previa (siempre se llega desde el mismo lado); None = no
paso_rampa_1t     = 0.005e-6                    # A
espera_I_1t       = 0.2                      # s, espera después de llegar a cada corriente
volver_a_1t       = None                     # A, corriente al terminar (None = queda en la última)

# ---- VNA ----
centro_busqueda_1t = 6.07e9                 # Hz (res_q1)
span_busqueda_1t   = 30e6                    # Hz
puntos_busqueda_1t = 201
ifbw_busqueda_1t   = 10                      # Hz
p_vna_1t           = -20                     # dBm (búsqueda y CW)
puntos_cw_1t       = 3                     # puntos CW promediados en cada corriente
ifbw_cw_1t         = 1                     # Hz -> puntos_cw_1t / ifbw_cw_1t = 1 s por corriente
buscar_cada_1t     = 0                       # 0 = f CW fija (mínimo de la primera corriente); n = cada n corrientes
f_cw_1t            = None                    # Hz; si se da, f CW fija ahí y no se busca nunca

# ---- metadata ----
resonador_1t = 'res_q2'
qubit_1t     = 'Q2'
notas_1t     = 'un tono: barrido fino de corriente, S21 en CW, sin drive'

n_b = 0 if f_cw_1t is not None else (1 if buscar_cada_1t == 0 else int(np.ceil(len(corrientes_1t) / buscar_cada_1t)))
t1 = tiempo_estimado_un_tono(len(corrientes_1t), puntos_cw_1t, ifbw_cw_1t, espera_I_1t, n_b,
                             puntos_busqueda_1t, ifbw_busqueda_1t)
print(f'{len(corrientes_1t)} corrientes de {corrientes_1t[0]*1e6:.3f} a {corrientes_1t[-1]*1e6:.3f} uA, '
      f'{n_b} búsqueda(s) -> ~{t1 / 60:.1f} min')

399 corrientes de -0.005 a -1.995 uA, 1 búsqueda(s) -> ~22.3 min


In [18]:
res_1t = medir_un_tono(
    pna, fuente, corrientes_1t,
    centro_busqueda=centro_busqueda_1t, span_busqueda=span_busqueda_1t,
    puntos_busqueda=puntos_busqueda_1t, ifbw_busqueda=ifbw_busqueda_1t, p_vna=p_vna_1t,
    puntos_cw=puntos_cw_1t, ifbw_cw=ifbw_cw_1t, f_cw=f_cw_1t, buscar_cada=buscar_cada_1t,
    paso_rampa=paso_rampa_1t, espera_I=espera_I_1t, preparar_desde=preparar_desde_1t, volver_a=volver_a_1t,
    sg=sg, vivo=True, nombre=f'un_tono_{qubit_1t}',
    resonador=resonador_1t, qubit=qubit_1t, att_vna=globals().get('att_vna', ''), notas=notas_1t)
print('run:', res_1t['run_id'])

un tono 20261111_165717: 399 corrientes, f CW fija (mínimo buscado en la primera corriente), 1 búsqueda(s) -> ~22.3 min (sin la rampa de la bobina)
preparando: rampa a -1.99 uA
Starting experimental run with id: 238. 
I =   -0.0050 uA | mínimo 6.069850 GHz (-72.99 dB) | f CW 6.069850 GHz
  19/399  I =   -0.0950 uA  |S21|  -72.57 dB  fase  -56.88°  | 2.1 min, faltan ~42.2 min
  38/399  I =   -0.1900 uA  |S21|  -72.55 dB  fase  -57.27°  | 3.2 min, faltan ~29.9 min
  57/399  I =   -0.2850 uA  |S21|  -72.48 dB  fase  -57.12°  | 4.2 min, faltan ~25.1 min
  76/399  I =   -0.3800 uA  |S21|  -72.50 dB  fase  -57.57°  | 5.2 min, faltan ~22.3 min
  95/399  I =   -0.4750 uA  |S21|  -72.52 dB  fase  -58.11°  | 6.3 min, faltan ~20.1 min
  114/399  I =   -0.5700 uA  |S21|  -72.66 dB  fase  -57.35°  | 7.3 min, faltan ~18.3 min
  133/399  I =   -0.6650 uA  |S21|  -72.63 dB  fase  -57.69°  | 8.3 min, faltan ~16.7 min
  152/399  I =   -0.7600 uA  |S21|  -72.64 dB  fase  -58.48°  | 9.4 min, faltan ~15.2 

In [7]:
# cargar y graficar un run de un tono (cualquier run_id)
run_1t = res_1t['run_id']
d1 = cargar_un_tono(run_1t)
I_uA = d1['I'] * 1e6
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(I_uA, 20 * np.log10(np.abs(d1['z'])), '.-', ms=3, lw=.6)
ax[0].set_ylabel('|S21| CW [dB]')
ax[1].plot(I_uA, np.degrees(np.unwrap(np.angle(d1['z']))), '.-', ms=3, lw=.6, color='C3')
ax[1].set_ylabel('fase CW [°]')
ok = np.isfinite(d1['f_min'])
if ok.any():
    ax[2].plot(I_uA[ok], (d1['f_min'][ok] - d1['f_min'][ok][0]) / 1e3, 'o-', ms=3, lw=.6)
ax[2].plot(I_uA, (d1['f_cw'] - d1['f_cw'][0]) / 1e3, 'k:', lw=.8, label='f CW')
ax[2].set_ylabel('f − primera [kHz]'); ax[2].legend(fontsize=8)
ax[-1].set_xlabel('I bobina [uA]')
for a in ax:
    a.grid(alpha=.3)
ax[0].set_title(f"run {run_1t} | f CW inicial {d1['f_cw'][0]/1e9:.6f} GHz | {d1['metadata'].get('resonador', '')}")
fig.tight_layout()
plt.show()

### Cerrar

In [8]:
sg.rf_output(False)
# fuente.rampa(0.0, paso=1e-6)      # descomentá para bajar la corriente de la bobina a 0
pna.close()
sg.close()
fuente.inst.close()